# Resultados automáticos de extracción de claims

Este notebook construye las tablas comparativas de la primera fase del experimento, con el formato de una tabla de paper. Lee los `summary.csv` producidos por `scripts/evaluate_claims.py` para los cuatro modelos locales:

- T5S
- BARTO
- mBART-50
- uMT5-Base

La evaluación es automática y de referencia: METEOR, ROUGE-L F1 y exact match normalizado. No incluye revisión humana ni métricas de acuerdo entre anotadores.

## Interpretación y límites

Las métricas de referencia comparan cada predicción con `reference_claim`, que debe estar anotado antes de ejecutar la evaluación. No miden por sí solas la verdad médica del claim.

Las tasas de salida vacía, claims de menos de cinco tokens, longitud media y duplicados son diagnósticos operativos; se reportan aparte y no se presentan como calidad factual.

Si todavía no existen resultados, el notebook muestra qué falta y no inventa valores.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display


def find_repository_root(start: Path | None = None) -> Path:
    """Encuentra la raíz desde el directorio de ejecución del notebook."""
    start = (start or Path.cwd()).resolve()
    candidates = (start, *start.parents)
    for candidate in candidates:
        if (candidate / 'configs' / 'models.json').is_file() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError(
        'No se encontró la raíz del repositorio. Abre Jupyter desde el repositorio o desde notebooks/.'
    )


ROOT = find_repository_root()
EVALUATION_ROOT = ROOT / 'artifacts' / 'evaluation'
TABLES_ROOT = EVALUATION_ROOT / 'tables'
TABLES_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Raíz del repositorio: {ROOT}')
print(f'Directorio de evaluación: {EVALUATION_ROOT}')
print(f'Se guardarán las tablas en: {TABLES_ROOT}')

In [ ]:
# Carga el registro para conservar el orden y los nombres oficiales de los modelos.
registry_payload = json.loads((ROOT / 'configs' / 'models.json').read_text(encoding='utf-8'))
registry = pd.DataFrame(
    [
        {
            'model_id': item['id'],
            'model_display_name': item['display_name'],
            'base_checkpoint': item['base_checkpoint'],
            'paper': item['paper'],
        }
        for item in registry_payload['models']
    ]
)


def discover_summary_files(root: Path) -> list[Path]:
    files: list[Path] = []
    direct = root / 'summary.csv'
    if direct.is_file():
        files.append(direct)
    files.extend(sorted(path for path in root.glob('*/summary.csv') if path.is_file()))
    return list(dict.fromkeys(files))


summary_files = discover_summary_files(EVALUATION_ROOT)
frames: list[pd.DataFrame] = []
for path in summary_files:
    frame = pd.read_csv(path)
    if frame.empty:
        continue
    if 'model_id' not in frame.columns:
        frame['model_id'] = path.parent.name
    frame['summary_source'] = str(path.relative_to(ROOT))
    frames.append(frame)

if frames:
    summaries = pd.concat(frames, ignore_index=True)
    # Si existe una salida directa y otra por modelo, conserva una sola fila por modelo.
    summaries = summaries.drop_duplicates(subset=['model_id'], keep='first')
else:
    summaries = pd.DataFrame(columns=['model_id'])

display_names = dict(zip(registry['model_id'], registry['model_display_name']))
if 'model_display_name' not in summaries.columns:
    summaries['model_display_name'] = summaries['model_id'].map(display_names)
else:
    summaries['model_display_name'] = summaries['model_display_name'].fillna(
        summaries['model_id'].map(display_names)
    )

order = {model_id: position for position, model_id in enumerate(registry['model_id'])}
if not summaries.empty:
    summaries['_order'] = summaries['model_id'].map(order).fillna(len(order))
    summaries = summaries.sort_values(['_order', 'model_id']).drop(columns='_order').reset_index(drop=True)

status = registry[['model_id', 'model_display_name', 'base_checkpoint', 'paper']].copy()
status['summary_found'] = status['model_id'].isin(set(summaries['model_id']))

display(Markdown('### Estado de los resultados'))
display(status)
if summaries.empty:
    display(Markdown(
        'Todavía no hay `summary.csv`. Primero entrena, infiere y evalúa los modelos; al volver a ejecutar este notebook se llenarán las tablas.'
    ))
else:
    display(Markdown(f'Se encontraron **{len(summaries)}** modelo(s) evaluado(s) en **{len(summary_files)}** archivo(s) de resumen.'))
    display(summaries)

## Tabla 1. Métricas automáticas de extracción

La negrita marca el mejor valor entre los modelos disponibles. Las flechas indican si un valor mayor (`↑`) o menor (`↓`) es preferible. `N ref.` es el número de predicciones que sí pudieron emparejarse con un `reference_claim`.

In [ ]:
def highlight_extrema(data: pd.DataFrame, columns: list[str], maximize: bool) -> pd.DataFrame:
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for column in columns:
        if column not in data.columns:
            continue
        values = pd.to_numeric(data[column], errors='coerce')
        if values.notna().any():
            best = values.max() if maximize else values.min()
            styles.loc[values.eq(best), column] = 'font-weight: bold'
    return styles


metric_columns = ['meteor', 'rouge_l_f1', 'exact_match']
for column in ['model_display_name', 'prediction_n', 'reference_n', *metric_columns]:
    if column not in summaries.columns:
        summaries[column] = pd.NA

table_automatic = summaries[
    ['model_display_name', 'prediction_n', 'reference_n', 'meteor', 'rouge_l_f1', 'exact_match']
].copy()
table_automatic = table_automatic.rename(
    columns={
        'model_display_name': 'Modelo',
        'prediction_n': 'N pred.',
        'reference_n': 'N ref.',
        'meteor': 'METEOR ↑',
        'rouge_l_f1': 'ROUGE-L F1 ↑',
        'exact_match': 'Exact match ↑',
    }
)

table_automatic_style = (
    table_automatic.style
    .format(
        {
            'METEOR ↑': '{:.3f}',
            'ROUGE-L F1 ↑': '{:.3f}',
            'Exact match ↑': '{:.3f}',
        },
        na_rep='—',
    )
    .apply(
        lambda data: highlight_extrema(
            data, ['METEOR ↑', 'ROUGE-L F1 ↑', 'Exact match ↑'], maximize=True
        ),
        axis=None,
    )
    .set_caption('Resultados automáticos contra claims de referencia')
)
display(table_automatic_style)
table_automatic.to_csv(TABLES_ROOT / 'table_1_automatic_metrics.csv', index=False)
(TABLES_ROOT / 'table_1_automatic_metrics.html').write_text(
    table_automatic_style.to_html(), encoding='utf-8'
)
print(f'Tabla guardada en: {TABLES_ROOT / "table_1_automatic_metrics.csv"}')

## Tabla 2. Diagnósticos operativos

Estos indicadores sirven para detectar fallos de generación o salidas degeneradas. No sustituyen METEOR, ROUGE-L ni exact match, y no deben interpretarse como fidelidad o veracidad.

In [ ]:
diagnostic_columns = [
    'empty_rate',
    'shorter_than_5_tokens_rate',
    'mean_prediction_tokens_operational',
    'duplicate_output_rate',
]
for column in ['model_display_name', *diagnostic_columns]:
    if column not in summaries.columns:
        summaries[column] = pd.NA

table_diagnostics = summaries[
    ['model_display_name', *diagnostic_columns]
].copy().rename(
    columns={
        'model_display_name': 'Modelo',
        'empty_rate': 'Salidas vacías ↓',
        'shorter_than_5_tokens_rate': 'Menos de 5 tokens ↓',
        'mean_prediction_tokens_operational': 'Tokens medios',
        'duplicate_output_rate': 'Duplicados ↓',
    }
)

table_diagnostics_style = (
    table_diagnostics.style
    .format(
        {
            'Salidas vacías ↓': '{:.3f}',
            'Menos de 5 tokens ↓': '{:.3f}',
            'Tokens medios': '{:.2f}',
            'Duplicados ↓': '{:.3f}',
        },
        na_rep='—',
    )
    .apply(
        lambda data: highlight_extrema(
            data,
            ['Salidas vacías ↓', 'Menos de 5 tokens ↓', 'Duplicados ↓'],
            maximize=False,
        ),
        axis=None,
    )
    .set_caption('Diagnósticos operativos de la generación')
)
display(table_diagnostics_style)
table_diagnostics.to_csv(TABLES_ROOT / 'table_2_operational_diagnostics.csv', index=False)
(TABLES_ROOT / 'table_2_operational_diagnostics.html').write_text(
    table_diagnostics_style.to_html(), encoding='utf-8'
)
print(f'Tabla guardada en: {TABLES_ROOT / "table_2_operational_diagnostics.csv"}')

## Lectura final

Para el informe, usa la Tabla 1 como comparación principal. La Tabla 2 documenta problemas de ejecución y repetición. No se incluye una matriz de confusión porque la tarea es generación de texto, no clasificación binaria.

Este notebook tampoco afirma que un claim sea verdadero o falso: solo compara la salida generada con el claim de referencia y documenta el comportamiento operativo de cada modelo.

In [ ]:
expected_models = set(registry['model_id'])
available_models = set(summaries['model_id'])
missing_models = [model_id for model_id in registry['model_id'] if model_id not in available_models]

if missing_models:
    display(Markdown(
        '**Pendientes:** todavía no hay resultados para: '
        + ', '.join(f'`{model_id}`' for model_id in missing_models)
        + '. Ejecuta su entrenamiento, inferencia y evaluación, y vuelve a correr el notebook.'
    ))
else:
    display(Markdown('**Listo:** los cuatro modelos tienen un resumen de evaluación disponible.'))

print('Archivos generados:')
for path in sorted(TABLES_ROOT.glob('table_*')):
    print(f'- {path.relative_to(ROOT)}')